## BERT의 사전 학습 과정 재현하기

In [7]:
# Pytorch로 Bert 구현하기
import math
import torch
import torch.nn as nn


class BertConfig:
    def __init__(self, vocab_size, hidden_size=384, num_layers=6, num_heads=6,
                 intermediate_size=1536, max_position=64, dropout=0.1):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.intermediate_size = intermediate_size
        self.max_position = max_position
        self.dropout = dropout


class BertEmbeddings(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(cfg.max_position, cfg.hidden_size)
        self.seg_emb = nn.Embedding(2, cfg.hidden_size)  # 문장 A/B 구분 (MLM만 할 땐 전부 0)
        self.norm = nn.LayerNorm(cfg.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, input_ids, token_type_ids=None):
        pos = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        x = self.token_emb(input_ids) + self.pos_emb(pos) + self.seg_emb(token_type_ids)
        return self.dropout(self.norm(x))


class BertModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embeddings = BertEmbeddings(cfg)
        # norm_first=True (Pre-LN): 원 논문 BERT는 Post-LN이지만, Post-LN은 수렴이 느리고
        # 긴 warmup이 필요해서 짧은 학습 예산에서는 Pre-LN이 훨씬 빠르게 수렴한다.
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.hidden_size, nhead=cfg.num_heads,
            dim_feedforward=cfg.intermediate_size, dropout=cfg.dropout,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, cfg.num_layers,
            norm=nn.LayerNorm(cfg.hidden_size, eps=1e-12),  # Pre-LN은 마지막에 norm 필요
        )

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        x = self.embeddings(input_ids, token_type_ids)
        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        return self.encoder(x, src_key_padding_mask=pad_mask)  # (B, T, hidden)


class BertForMaskedLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.bert = BertModel(cfg)
        self.transform = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.hidden_size),
            nn.GELU(),
            nn.LayerNorm(cfg.hidden_size, eps=1e-12),
        )
        self.decoder = nn.Linear(cfg.hidden_size, cfg.vocab_size)
        self.decoder.weight = self.bert.embeddings.token_emb.weight  # weight tying

        # BERT식 초기화 (std=0.02) — 매우 중요!
        # nn.Embedding 기본 초기화는 N(0,1)인데, weight tying 때문에 이 행렬이 출력층
        # 가중치로도 쓰여 logits std가 ~16까지 커지고 초기 loss가 40+에서 시작한다.
        # (정상적인 무작위 추측 loss는 ln(vocab)=10.3) N(0,0.02)로 줄이면 logits std≈0.3.
        emb = self.bert.embeddings
        for e in (emb.token_emb, emb.pos_emb, emb.seg_emb):
            nn.init.normal_(e.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            emb.token_emb.weight[emb.token_emb.padding_idx].fill_(0)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        hidden = self.bert(input_ids, attention_mask, token_type_ids)
        return self.decoder(self.transform(hidden))  # (B, T, vocab) 각 위치의 토큰 예측 logits

In [8]:
# https://huggingface.co/datasets/Skylion007/openwebtext
# 데이터를 MLM 데이터셋으로 구축

# 전체가 38GB라 streaming으로 앞부분 문서만 가져오고
# 문서들을 토큰화해 이어붙인 뒤 64토큰 블록으로 잘라 시퀀스 생성
# - MAX_LEN 128->64: 같은 토큰 수로 스텝 수 2배, attention 비용 1/4,
#   짧은 데모 문장과의 길이 분포 차이도 줄어듦
# - 문서를 모아 배치 토큰화 (fast tokenizer는 배치 입력이 훨씬 빠름)
%pip install -q datasets

from datasets import load_dataset
from transformers import BertTokenizerFast
from torch.utils.data import Dataset, DataLoader

# BERT의 WordPiece 토크나이저 재사용 ([MASK], [CLS], [SEP] 토큰 포함, pad=0)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

MAX_LEN = 64
NUM_DOCS = 30_000  # 약 30M 토큰
MLM_PROB = 0.15
BATCH_DOCS = 512   # 배치 토큰화 단위

stream = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

body = MAX_LEN - 2  # [CLS], [SEP] 자리 제외
blocks, buffer, texts = [], [], []

def tokenize_batch(texts):
    for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]:
        buffer.extend(ids)
        while len(buffer) >= body:
            blocks.append([tokenizer.cls_token_id] + buffer[:body] + [tokenizer.sep_token_id])
            del buffer[:body]

for i, doc in enumerate(stream):
    if i >= NUM_DOCS:
        break
    texts.append(doc["text"])
    if len(texts) == BATCH_DOCS:
        tokenize_batch(texts)
        texts = []
    if (i + 1) % 5000 == 0:
        print(f"{i+1:,} docs -> {len(blocks):,} sequences")
if texts:
    tokenize_batch(texts)

# 고정 길이라 리스트 대신 하나의 텐서로 보관 (메모리 절약 + 빠른 인덱싱)
data = torch.tensor(blocks, dtype=torch.long)
del blocks
print(f"총 {data.size(0):,}개 시퀀스 (길이 {MAX_LEN})")

# 학습에 쓰지 않을 held-out 블록 분리 (마스크 정확도 평가용)
VAL_SIZE = 2000
val_data, train_data = data[:VAL_SIZE], data[VAL_SIZE:]


class MLMDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        input_ids = self.data[idx].clone()
        labels = input_ids.clone()

        # 1) 전체 토큰의 15%를 예측 대상으로 선택 ([CLS]/[SEP]인 양 끝은 제외)
        prob = torch.full(input_ids.shape, MLM_PROB)
        prob[0] = prob[-1] = 0.0
        masked = torch.bernoulli(prob).bool()
        labels[~masked] = -100  # 선택 안 된 위치는 loss 계산에서 제외

        # 2) 선택된 토큰 중 80%는 [MASK]로 교체
        replace = torch.bernoulli(torch.full(input_ids.shape, 0.8)).bool() & masked
        input_ids[replace] = tokenizer.mask_token_id

        # 3) 10%는 랜덤 토큰으로 교체, 나머지 10%는 원본 유지
        rand = torch.bernoulli(torch.full(input_ids.shape, 0.5)).bool() & masked & ~replace
        input_ids[rand] = torch.randint(tokenizer.vocab_size, input_ids.shape)[rand]

        return {"input_ids": input_ids, "labels": labels}


train_loader = DataLoader(MLMDataset(train_data), batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(MLMDataset(val_data), batch_size=256, shuffle=False)

# 마스킹 결과 확인
sample = MLMDataset(train_data)[0]
print("\n[마스킹 예시]")
print(tokenizer.decode(sample["input_ids"][:40]))
print("예측 대상 토큰 수:", (sample["labels"] != -100).sum().item(), f"/ {MAX_LEN}")

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 512). Running this sequence through the model will result in indexing errors


5,000 docs -> 81,144 sequences
10,000 docs -> 169,160 sequences
15,000 docs -> 259,605 sequences
20,000 docs -> 350,040 sequences
25,000 docs -> 433,125 sequences
30,000 docs -> 524,212 sequences
총 529,337개 시퀀스 (길이 64)

[마스킹 예시]
[CLS] no [MASK] that had the racial disposition of this case [MASK] reversed this would [MASK] reported as an act of terror [MASK] - dr tarlochan singh [MASK] [MASK] [MASK], sarandev [MASK] [MASK] brother [MASK]
예측 대상 토큰 수: 15 / 64


In [9]:
# 모델 학습 단계
# BERT-small급(6 layer, hidden 384, ~23M)을 MLM objective로 사전학습합니다.
# loss는 마스킹된 위치(-100이 아닌 label)에서만 계산됩니다.
# 정상 기준: loss 10.3 근처에서 시작 -> 종료 시 4~5대, held-out 마스크 top-1 정확도 30%+
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=384, num_layers=6, num_heads=6,
    intermediate_size=1536, max_position=MAX_LEN,
)
model = BertForMaskedLM(cfg).to(device)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M | device: {device}")

EPOCHS = 2       # dynamic masking이라 epoch마다 다른 마스크로 학습됨
LOG_EVERY = 200

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# linear warmup + linear decay
total_steps = len(train_loader) * EPOCHS
warmup_steps = max(50, int(total_steps * 0.06))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: s / warmup_steps if s < warmup_steps
    else max(0.0, (total_steps - s) / (total_steps - warmup_steps)),
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)


# held-out 블록에서 마스크된 토큰 top-1 정확도 측정
# fill_mask는 눈으로 보는 정성 평가라, 문맥 학습 여부를 수치로 확인하는 용도
@torch.no_grad()
def masked_accuracy(loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        preds = model(input_ids).argmax(-1)
        m = labels != -100
        correct += (preds[m] == labels[m]).sum().item()
        total += m.sum().item()
    model.train()
    return correct / total


model.train()
step, running_loss, t0 = 0, 0.0, time.time()
for epoch in range(EPOCHS):
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(input_ids)
            loss = criterion(logits.view(-1, cfg.vocab_size), labels.view(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        step += 1
        if step % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            print(f"step {step}/{total_steps} | mlm loss {avg:.4f} | ppl {math.exp(avg):.1f} "
                  f"| lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
            running_loss = 0.0

    acc = masked_accuracy(val_loader)
    print(f"== epoch {epoch+1} 종료 | held-out 마스크 top-1 정확도: {acc:.3f} ==")

print("학습 완료")


# 학습 결과 확인: 문장의 [MASK]를 모델이 어떻게 채우는지 top-k 예측 출력
@torch.no_grad()
def fill_mask(text, topk=5):
    model.eval()
    enc = tokenizer(text, return_tensors="pt").to(device)
    logits = model(enc["input_ids"], enc["attention_mask"])
    print(f"\n> {text}")
    for pos in (enc["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]:
        probs = logits[0, pos].softmax(-1)  # 전체 vocab 기준 확률
        top = probs.topk(topk)
        preds = [f"{tokenizer.decode([t])} ({p:.2f})"
                 for t, p in zip(top.indices.tolist(), top.values.tolist())]
        print("  [MASK] ->", ", ".join(preds))

fill_mask("The capital of France is [MASK].")
fill_mask("I went to the [MASK] to buy some milk.")
fill_mask("She [MASK] a book about the history of science.")
fill_mask("He drove his [MASK] to work this morning.")

params: 22.57M | device: cuda


/tmp/ipykernel_4585/948663782.py:47: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


step 200/8240 | mlm loss 8.7873 | ppl 6550.3 | lr 1.21e-04 | 28s
step 400/8240 | mlm loss 7.1833 | ppl 1317.2 | lr 2.43e-04 | 57s
step 600/8240 | mlm loss 6.9097 | ppl 1001.9 | lr 2.96e-04 | 87s
step 800/8240 | mlm loss 6.7507 | ppl 854.7 | lr 2.88e-04 | 117s
step 1000/8240 | mlm loss 6.6473 | ppl 770.7 | lr 2.80e-04 | 146s
step 1200/8240 | mlm loss 6.4523 | ppl 634.1 | lr 2.73e-04 | 176s
step 1400/8240 | mlm loss 6.1103 | ppl 450.5 | lr 2.65e-04 | 206s
step 1600/8240 | mlm loss 5.7835 | ppl 324.9 | lr 2.57e-04 | 236s
step 1800/8240 | mlm loss 5.5586 | ppl 259.5 | lr 2.49e-04 | 265s
step 2000/8240 | mlm loss 5.3974 | ppl 220.8 | lr 2.42e-04 | 295s
step 2200/8240 | mlm loss 5.2680 | ppl 194.0 | lr 2.34e-04 | 325s
step 2400/8240 | mlm loss 5.1787 | ppl 177.5 | lr 2.26e-04 | 354s
step 2600/8240 | mlm loss 5.0761 | ppl 160.1 | lr 2.18e-04 | 384s
step 2800/8240 | mlm loss 5.0260 | ppl 152.3 | lr 2.11e-04 | 414s
step 3000/8240 | mlm loss 4.9741 | ppl 144.6 | lr 2.03e-04 | 444s
step 3200/8240